# BestTime API minimal healthcare venue forecast pull

BestTime's recommended pattern is to create or refresh a venue forecast only once every few weeks, then query existing forecasts between refreshes. Live foot-traffic calls are hourly and more expensive for this healthcare venue coverage task.

This notebook uses only Python standard library modules, so it does not require `pandas` or `requests`.

## Credit plan

- Endpoint: `POST https://besttime.app/api/v1/forecasts`
- Input: `venue_name` + `venue_address`
- Successful new forecast: 2 credits per venue
- Unsuccessful forecast: 1 credit per venue
- 600 monthly credits covers about 300 successful venue forecasts
- 1083 healthcare venues would require about 2166 credits for one complete refresh

In [6]:
from __future__ import annotations

import csv
import json
import os
import time
import urllib.parse
import urllib.request
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Locate project root by walking up until we find docker-compose.yml
_here = Path.cwd().resolve()
PROJECT_ROOT = _here
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / "docker-compose.yml").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f"Working directory set to project root: {PROJECT_ROOT}")

# Plain-text API import requested for this experiment.
# Do not commit a real private key to Git.
API_KEY_PRIVATE = "pri_5ef7704dfc534c4eb42b9d590d9a5e7f"

# Safety switch: keep True for planning/log validation. Set False to call BestTime.
DRY_RUN = False

# Set True once when you want to clear previous BestTime logs/raw outputs before rebuilding the queue.
# After the reset run, set it back to False to avoid deleting new results.
RESET_OUTPUTS = False

TARGET_TOTAL = 1083
MONTHLY_CREDIT_BUDGET = 600
MAX_CREDITS_THIS_RUN = 600
CREDITS_PER_SUCCESS = 2
CREDITS_PER_UNSUCCESSFUL = 1

SLEEP_SECONDS_BETWEEN_CALLS = 0
REQUEST_TIMEOUT_SECONDS = 8

BASE_DIR = Path("Data+ML/test/6.28-7.3")
INPUT_CSV = Path("Data+ML/test/6.22-6.27/output/healthcare_prediction_group_members.csv")
OUTPUT_DIR = BASE_DIR / "output"
RAW_DIR = OUTPUT_DIR / "besttime_forecast_raw"
CALL_LOG_JSONL = OUTPUT_DIR / "besttime_api_call_log.jsonl"
CALL_LOG_CSV = OUTPUT_DIR / "besttime_api_call_log.csv"
SUMMARY_CSV = OUTPUT_DIR / "besttime_forecast_summary.csv"

FORECAST_URL = "https://besttime.app/api/v1/forecasts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)

if RESET_OUTPUTS:
    for path in [CALL_LOG_JSONL, CALL_LOG_CSV, SUMMARY_CSV]:
        if path.exists():
            path.unlink()
            print(f"Deleted {path}")
    for path in RAW_DIR.glob("*.json"):
        path.unlink()
    print(f"Cleared raw BestTime JSON files in {RAW_DIR}")

print("Configured BestTime healthcare forecast pull")
print(f"DRY_RUN={DRY_RUN}")
print(f"Input CSV: {INPUT_CSV.resolve()}")
print(f"Output dir: {OUTPUT_DIR.resolve()}")

Working directory set to project root: /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project
Configured BestTime healthcare forecast pull
DRY_RUN=False
Input CSV: /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.22-6.27/output/healthcare_prediction_group_members.csv
Output dir: /Users/alex/Documents/COMP47360-Research_Practicum/Group6_Summer-Project/Data+ML/test/6.28-7.3/output


In [7]:
def read_csv_dicts(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))


def write_csv_dicts(path: Path, rows: list[dict[str, Any]]) -> None:
    if not rows:
        return
    fieldnames: list[str] = []
    for row in rows:
        for key in row.keys():
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def first_non_empty(row: dict[str, Any], columns: list[str]) -> str:
    for col in columns:
        value = str(row.get(col, "") or "").strip()
        if value:
            return value
    return ""


def build_venue_address(row: dict[str, Any]) -> str:
    address = first_non_empty(row, ["address", "venue_address", "external_address", "street_address"])
    if address:
        return address

    lat = first_non_empty(row, ["latitude", "lat"])
    lng = first_non_empty(row, ["longitude", "lng", "lon"])
    if lat and lng:
        return f"{lat}, {lng}, New York, NY"

    district = first_non_empty(row, ["district", "borough"])
    return f"{district}, New York, NY".strip(", ")


raw_rows = read_csv_dicts(INPUT_CSV)
seen: set[str] = set()
venues: list[dict[str, Any]] = []

for row in raw_rows:
    venue_id = row.get("venue_id", "").strip()
    if not venue_id or venue_id in seen:
        continue
    seen.add(venue_id)
    row = dict(row)
    row["besttime_venue_name"] = first_non_empty(row, ["name", "venue_name", "external_name"])
    row["besttime_venue_address"] = build_venue_address(row)
    venues.append(row)
    if len(venues) >= TARGET_TOTAL:
        break

missing_name = sum(1 for row in venues if not row["besttime_venue_name"])
missing_address = sum(1 for row in venues if not row["besttime_venue_address"])

print(f"Loaded healthcare venues: {len(venues)}")
print(f"Missing names: {missing_name}")
print(f"Missing addresses: {missing_address}")
for row in venues[:5]:
    print(row["venue_id"], "|", row["besttime_venue_name"], "|", row["besttime_venue_address"])

Loaded healthcare venues: 1083
Missing names: 0
Missing addresses: 0
0006c1aef358bc9d07ac1a05258679fbfa71 | Columbia University Health Care Mobile Van | 40.8717, -73.91321, New York, NY
002dcde3e18169cb13938576b3c2e14e81fa | Wondercare Inc. | 40.750059, -73.98858, New York, NY
0037f33af41eadfc1e21c27e139fbbe1fe7b | The Dental Office of Dr. Ronald Birnbaum | 40.7684614, -73.9865467, New York, NY
004dab5b88281a1496b609a6438341c37e5f | PS 108 | 40.795238, -73.948235, New York, NY
0059308dfe9e87f8242bd080e6ad280a5af0 | Matthew DelMauro | 40.7571347, -73.97853, New York, NY


In [8]:
def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists() or path.stat().st_size == 0:
        return []
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


existing_logs = read_jsonl(CALL_LOG_JSONL)
completed_states = {"success", "no_forecast", "api_error"}
completed_venue_ids = {row["local_venue_id"] for row in existing_logs if row.get("final_state") in completed_states}
credits_spent_logged = sum(int(row.get("estimated_credits") or 0) for row in existing_logs)

remaining_credit_budget = max(0, MAX_CREDITS_THIS_RUN - credits_spent_logged)
max_successful_calls_this_run = remaining_credit_budget // CREDITS_PER_SUCCESS
queue = [row for row in venues if row["venue_id"] not in completed_venue_ids][:max_successful_calls_this_run]

full_refresh_credits = len(venues) * CREDITS_PER_SUCCESS
monthly_success_capacity = MONTHLY_CREDIT_BUDGET // CREDITS_PER_SUCCESS

print(f"Target venues: {len(venues)}")
print(f"Full one-time refresh estimate: {full_refresh_credits} credits")
print(f"Monthly 600-credit success capacity: {monthly_success_capacity} venues")
print(f"Already logged estimated credits: {credits_spent_logged}")
print(f"Remaining credits allowed this run: {remaining_credit_budget}")
print(f"Queued venues this run: {len(queue)}")

Target venues: 1083
Full one-time refresh estimate: 2166 credits
Monthly 600-credit success capacity: 300 venues
Already logged estimated credits: 77
Remaining credits allowed this run: 523
Queued venues this run: 261


In [9]:
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def append_jsonl(path: Path, row: dict[str, Any]) -> None:
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def classify_response(payload: dict[str, Any], http_status) -> tuple[str, int]:
    if http_status is None:
        return "request_exception", 0
    if http_status >= 500:
        return "server_error", 0

    status = str(payload.get("status", "")).lower()
    venue_forecasted = payload.get("venue_forecasted")
    if status == "ok" and venue_forecasted is not False:
        return "success", CREDITS_PER_SUCCESS
    if status == "ok" and venue_forecasted is False:
        return "no_forecast", CREDITS_PER_UNSUCCESSFUL
    return "api_error", CREDITS_PER_UNSUCCESSFUL


def post_besttime_forecast(params: dict[str, str]):
    query = urllib.parse.urlencode(params)
    url = f"{FORECAST_URL}?{query}"
    req = urllib.request.Request(url, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=REQUEST_TIMEOUT_SECONDS) as response:
            raw = response.read().decode("utf-8", errors="replace")
            return response.status, json.loads(raw), None
    except urllib.error.HTTPError as exc:
        raw = exc.read().decode("utf-8", errors="replace")
        try:
            payload = json.loads(raw)
        except Exception:
            payload = {"status": "Error", "raw_text": raw[:2000]}
        return exc.code, payload, None
    except Exception as exc:
        return None, {"status": "RequestException", "error": repr(exc)}, repr(exc)


def summarize_payload(local_venue_id: str, payload: dict[str, Any]) -> dict[str, Any]:
    info = payload.get("venue_info", {}) or {}
    analysis = payload.get("analysis", []) or []
    day_raw_hours = 0
    peak_hour_count = 0
    busy_hour_count = 0
    quiet_hour_count = 0

    if isinstance(analysis, list):
        for day in analysis:
            if isinstance(day, dict):
                day_raw = day.get("day_raw") or []
                day_raw_hours += len(day_raw) if isinstance(day_raw, list) else 0
                peak_hour_count += len(day.get("peak_hours") or [])
                busy_hour_count += len(day.get("busy_hours") or [])
                quiet_hour_count += len(day.get("quiet_hours") or [])

    return {
        "local_venue_id": local_venue_id,
        "besttime_venue_id": info.get("venue_id"),
        "besttime_venue_name": info.get("venue_name"),
        "besttime_venue_address": info.get("venue_address"),
        "besttime_venue_type": info.get("venue_type"),
        "venue_forecasted": payload.get("venue_forecasted"),
        "forecast_updated_on": payload.get("forecast_updated_on"),
        "analysis_days": len(analysis) if isinstance(analysis, list) else 0,
        "day_raw_hours": day_raw_hours,
        "peak_hour_count": peak_hour_count,
        "busy_hour_count": busy_hour_count,
        "quiet_hour_count": quiet_hour_count,
    }


print("Helper functions ready")

Helper functions ready


In [10]:
print("Starting BestTime forecast execution cell")
print(f"DRY_RUN={DRY_RUN}")
print(f"Queued venues={len(queue)}")
print(f"Max credits this run={MAX_CREDITS_THIS_RUN}")

if not DRY_RUN and API_KEY_PRIVATE == "PASTE_BESTTIME_PRIVATE_KEY_HERE":
    print("ERROR: API_KEY_PRIVATE is still the placeholder value.")
    raise ValueError("Set API_KEY_PRIVATE before running live calls.")

if not queue:
    print("No queued venues. Check existing logs or MAX_CREDITS_THIS_RUN.")

run_rows: list[dict[str, Any]] = []

for call_index, row in enumerate(queue, start=1):
    local_venue_id = row["venue_id"]
    params = {
        "api_key_private": API_KEY_PRIVATE,
        "venue_name": row["besttime_venue_name"],
        "venue_address": row["besttime_venue_address"],
    }

    started_at = utc_now_iso()
    raw_path = RAW_DIR / f"{local_venue_id}.json"

    print(f"[{call_index}/{len(queue)}] {local_venue_id} | {params['venue_name']} | dry_run={DRY_RUN}")

    if DRY_RUN:
        run_rows.append({
            "ts_utc": started_at,
            "dry_run": True,
            "local_venue_id": local_venue_id,
            "venue_name_input": params["venue_name"],
            "venue_address_input": params["venue_address"],
            "http_status": None,
            "besttime_status": "DRY_RUN",
            "final_state": "planned",
            "estimated_credits": 0,
            "raw_path": str(raw_path),
            "error": None,
        })
        continue

    http_status, payload, error = post_besttime_forecast(params)
    final_state, estimated_credits = classify_response(payload, http_status)
    raw_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")

    log_row = {
        "ts_utc": started_at,
        "dry_run": False,
        "local_venue_id": local_venue_id,
        "venue_name_input": params["venue_name"],
        "venue_address_input": params["venue_address"],
        "http_status": http_status,
        "besttime_status": payload.get("status"),
        "final_state": final_state,
        "estimated_credits": estimated_credits,
        "raw_path": str(raw_path),
        "error": error or payload.get("message") or payload.get("error"),
    }
    append_jsonl(CALL_LOG_JSONL, log_row)
    run_rows.append(log_row)
    print(f"    -> state={final_state}, http={http_status}, estimated_credits={estimated_credits}")
    time.sleep(SLEEP_SECONDS_BETWEEN_CALLS)

print(f"Rows processed in this notebook cell: {len(run_rows)}")
for row in run_rows[:5]:
    print(row)

Starting BestTime forecast execution cell
DRY_RUN=False
Queued venues=261
Max credits this run=600
[1/261] 0c1880368ff8214586570ebbf861286d5194 | Harlem Children's Zone Promise Academy 1 | dry_run=False
    -> state=request_exception, http=None, estimated_credits=0
[2/261] 0c1e6d486452d6c335d396d896bfbb8ece32 | Dermatology Specialists | dry_run=False
    -> state=request_exception, http=None, estimated_credits=0
[3/261] 0c263eeccd88d244c6f8eea88d7b5761a355 | Psychiatric Out-Patient Clinic | dry_run=False
    -> state=request_exception, http=None, estimated_credits=0
[4/261] 10d7bc69d4322530a7ac4a5c7fb3fe5ee5c9 | Intermediate School 218 | dry_run=False
    -> state=api_error, http=404, estimated_credits=1
[5/261] 118aede4a1689c78b011cdddde0d60f72703 | Drug Loft Pharmacy & Wellness Center | dry_run=False
    -> state=request_exception, http=None, estimated_credits=0
[6/261] 118b0dcb5c5b7e50e138d73c905d6ad9b784 | Russak Dermatology Clinic | dry_run=False
    -> state=api_error, http=404, 

In [11]:
if DRY_RUN:
    print(f"DRY RUN planned calls: {len(run_rows)}")
    print(f"Estimated max success credits if executed: {len(run_rows) * CREDITS_PER_SUCCESS}")
else:
    log_rows = read_jsonl(CALL_LOG_JSONL)
    write_csv_dicts(CALL_LOG_CSV, log_rows)

    summaries = []
    for raw_file in sorted(RAW_DIR.glob("*.json")):
        payload = json.loads(raw_file.read_text(encoding="utf-8"))
        summaries.append(summarize_payload(raw_file.stem, payload))
    write_csv_dicts(SUMMARY_CSV, summaries)

    estimated_credits = sum(int(row.get("estimated_credits") or 0) for row in log_rows)
    print(f"Log rows: {len(log_rows)}")
    print(f"Estimated credits logged: {estimated_credits}")
    print(f"Raw JSON files: {len(list(RAW_DIR.glob('*.json')))}")
    print(f"Call log CSV: {CALL_LOG_CSV}")
    print(f"Summary CSV: {SUMMARY_CSV}")

Log rows: 398
Estimated credits logged: 162
Raw JSON files: 329
Call log CSV: Data+ML/test/6.28-7.3/output/besttime_api_call_log.csv
Summary CSV: Data+ML/test/6.28-7.3/output/besttime_forecast_summary.csv


## Output files

- `output/besttime_api_call_log.jsonl`: append-only API call log for resume/audit
- `output/besttime_api_call_log.csv`: spreadsheet version of the call log
- `output/besttime_forecast_raw/*.json`: raw BestTime responses, one per local venue
- `output/besttime_forecast_summary.csv`: compact parsed forecast coverage summary

Run strategy for 1083 venues with 600 monthly credits: refresh about 300 venues per month, or about 75 venues per week if spreading the refresh budget across four weekly batches.